# AI Experiment Control Center — Google Colab Instrumentation Example

> **Important:** **Do not change your hyperparameters; these cells only observe.**
> Google Colab has no native execution-status or log API in ECC MVP, so Colab runs are monitored via `experiment_monitor` → W&B cloud plus your stored `notebook_url`.

## 1. Install `experiment-monitor`

Option A installs directly from GitHub (replace `<YOUR_GH_USER>`). Option B installs editable from a copy of `monitor/` uploaded to your Google Drive.

In [ ]:
# [Optional] Option A — only if the repo is on GitHub:
# Option A: Install directly from GitHub (replace <YOUR_GH_USER>)
# !pip install -q "git+https://github.com/<YOUR_GH_USER>/ai-experiment-control-center.git#subdirectory=monitor"

In [ ]:
# Option B (default): install the wheel from Google Drive
# Upload experiment_monitor-0.1.0-py3-none-any.whl to MyDrive/ecc/ once per Google account.
from google.colab import drive
drive.mount("/content/drive")
import glob, subprocess, sys
wheels = sorted(glob.glob("/content/drive/MyDrive/ecc/experiment_monitor-*.whl"))
assert wheels, "Wheel not found in MyDrive/ecc/"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", wheels[-1]])
print("Installed", wheels[-1])

## 2. Authenticate W&B from Colab Secrets

1. In Google Colab, open the **🔑 Secrets** panel on the left sidebar.
2. Click **Add new secret**, set **Name** = `WANDB_API_KEY` and **Value** = your personal W&B API key (from `https://wandb.ai/authorize`).
3. Turn ON the **Notebook access** toggle for `WANDB_API_KEY`.

Repeat this in each Colab Google account (`colab-1`, `colab-2`); both log into your single W&B workspace. `em.login_from_notebook_secrets("WANDB_API_KEY")` reads `google.colab.userdata` and never prints or returns the key.

In [ ]:
import experiment_monitor as em

ok = em.login_from_notebook_secrets("WANDB_API_KEY")
print("W&B authentication:", "OK" if ok else "FAILED — check Colab Secrets panel & Notebook access toggle")

## 3. Initialize ECC Monitor (`em.init`)

Keep your existing config dictionary untouched; pass `account="colab-1"` (or `"colab-2"`) as a non-sensitive label and paste your Colab notebook URL into `notebook_url=` so the dashboard's **Notebook ↗** button opens the right notebook.

In [ ]:
existing_config_dict = {
    "learning_rate": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "batch_size": 8,
    "epochs": 3,
}

# Set INIT_STANDALONE = True when running this cell against your real W&B project
INIT_STANDALONE = False
if INIT_STANDALONE:
    mon = em.init(
        project="shared-task-4",
        name="qwen3-vl-lora-r16-colab1",
        account="colab-1",
        notebook_url="https://colab.research.google.com/drive/<YOUR_COLAB_FILE_ID>",
        kaggle_kernel=None,
        model_name="Qwen/Qwen3-VL-4B-Instruct",
        seed=42,
        dataset_version="v2",
        dataset_split="train/val=90/10 seed42",
        prompt_strategy="en-prompt",
        eval_signature="v2-en-prompt-90-10-seed42",
        config=existing_config_dict,
    )

## 4A. Variant A — HuggingFace `Trainer` / TRL `SFTTrainer`

- **Do not change your hyperparameters; these cells only observe.**
- Attach `make_hf_callback(mon)` via `trainer.add_callback(make_hf_callback(mon))`. Works identically for `transformers.Trainer` and `trl.SFTTrainer`.
- **`report_to` note**: Set `report_to="none"` (recommended) so ECC logs metrics once under standardized `train/*` and `eval/*` keys. If you already use `report_to="wandb"`, `em.init(...)` reuses the active `wandb.run`.

In [ ]:
from experiment_monitor import make_hf_callback

USE_HF_TRAINER = False
if USE_HF_TRAINER:
    # Assume `trainer` is your existing HuggingFace Trainer or TRL SFTTrainer
    # configured with report_to="none" (recommended) and all your original hyperparameters.
    mon.track_best("eval/macro_f1", mode="max")
    trainer.add_callback(make_hf_callback(mon))
    with mon:
        trainer.train()

## 4B. Variant B — Plain PyTorch Loop (CPU-only Smoke Test)

- **Do not change your hyperparameters; these cells only observe.**
- Wrap your training loop in `with mon:` so exceptions and `KeyboardInterrupt` are captured with redacted error messages and written to `ecc_console.log`.
- Below is a tiny 5-step CPU-only dummy loop (`DRY_RUN = True`) with random tensors so you can smoke-test ECC integration without a GPU.

In [ ]:
# ECC-SMOKE
import os
import tempfile
import torch
import experiment_monitor as em

DRY_RUN = True
if DRY_RUN:
    torch.manual_seed(42)
    existing_config_dict = {
        "learning_rate": 1e-2,
        "batch_size": 4,
        "epochs": 1,
    }
    total_steps = 5
    model = torch.nn.Linear(8, 2)
    optimizer = torch.optim.SGD(model.parameters(), lr=existing_config_dict["learning_rate"])
    loss_fn = torch.nn.CrossEntropyLoss()

    mon = em.init(
        project="shared-task-4",
        name="smoke-pytorch-colab-1",
        account="colab-1",
        notebook_url="https://colab.research.google.com/drive/<YOUR_COLAB_FILE_ID>",
        model_name="dummy-linear-cpu",
        seed=42,
        dataset_version="smoke-v0",
        total_steps=total_steps,
        total_epochs=1.0,
        config=existing_config_dict,
    )

    with tempfile.TemporaryDirectory() as ckpt_dir:
        with mon:
            mon.track_best("eval/macro_f1", mode="max")
            for step in range(1, total_steps + 1):
                x = torch.randn(4, 8)
                y = torch.randint(0, 2, (4,))
                assert x.shape == (4, 8), f"Unexpected input shape: {x.shape}"

                optimizer.zero_grad()
                logits = model(x)
                assert logits.shape == (4, 2), f"Unexpected logits shape: {logits.shape}"
                loss = loss_fn(logits, y)
                loss.backward()
                optimizer.step()

                epoch = step / float(total_steps)
                mon.log(
                    {
                        "train/loss": float(loss.item()),
                        "train/lr": existing_config_dict["learning_rate"],
                        "train/epoch": epoch,
                    },
                    step=step,
                )
                mon.set_progress(step=step, total_steps=total_steps, epoch=epoch)

                if step == total_steps:
                    eval_f1 = 0.78
                    mon.log({"eval/loss": float(loss.item()), "eval/macro_f1": eval_f1}, step=step)
                    ckpt_path = os.path.join(ckpt_dir, f"checkpoint-step-{step}.pt")
                    torch.save(model.state_dict(), ckpt_path)
                    mon.log_checkpoint(ckpt_path, step=step, metric=eval_f1, metric_name="eval/macro_f1")
    print("ECC Colab smoke test finished cleanly.")

## 5. Resume an Interrupted Run (`resume_id=`)

If Colab disconnects a runtime, pass the previous W&B run ID to `resume_id=` to continue logging into the same run.

In [ ]:
RUN_RESUME_EXAMPLE = False
if RUN_RESUME_EXAMPLE:
    PREVIOUS_RUN_ID = "<YOUR_WANDB_RUN_ID>"  # e.g. 8-char W&B run id from the run URL
    with em.init(
        project="shared-task-4",
        name="qwen3-vl-lora-r16-colab1",
        account="colab-1",
        notebook_url="https://colab.research.google.com/drive/<YOUR_COLAB_FILE_ID>",
        model_name="Qwen/Qwen3-VL-4B-Instruct",
        seed=42,
        dataset_version="v2",
        resume_id=PREVIOUS_RUN_ID,
        config=existing_config_dict,
    ) as mon:
        # Resume your existing training loop / trainer.train(resume_from_checkpoint=True)
        pass